# Pipeline — Negociação Secundária de Crédito Privado

Cada bloco é um fluxo. Rode o que precisar isoladamente. Toda a lógica está em `scripts/pipeline_core.py`.

**Rode este notebook a partir da pasta `code/`.** Ordem e parâmetros de cada passo: ver `vault/11 - Pipeline de Execucao.md` e `vault/13 - Migracao Banco.md`.

## 0. Setup (rodar primeiro)

In [ ]:
import sys
from pathlib import Path
from datetime import date

scripts = Path.cwd() / "scripts"
if not scripts.exists():
    raise SystemExit(f"Rode o notebook a partir da pasta code/. cwd atual: {Path.cwd()}")
sys.path.insert(0, str(scripts))
import pipeline_core as pc

print("Hoje:", date.today())
print("Ultimos 5 dias uteis:", [d.isoformat() for d in pc.ultimos_n_dias_uteis(5)])

## 1. Rotina diária — últimos 5 dias úteis
Raspa + calcula os últimos 5 dias úteis (pega alterações retroativas) e gera o relatório no fim. É o que o `run_diario.py` chama no Task Scheduler.

In [ ]:
pc.run_ultimos_n(5)

## 2. Rodar uma única data de liquidação
Cadeia dos 13 passos para uma liquidação X (raspa X e X-1u).

In [ ]:
X = "2026-07-02"   # ajuste a data de liquidacao
pc.run_dia(X)

## 3. Setup inicial — rodar 1 vez no banco
Bootstrap da base: raspa o histórico largo de cada fonte (Anbima Data completa, deb/NTN-B ~4 meses, curva DI ~20 pregões, CRI/CRA ~5 pregões, boletim na janela escolhida) e roda a cadeia de cálculo. **Demorado.** Ajuste o início do boletim.

In [ ]:
# rodar_outstanding=True apenas no banco (terminal Bloomberg logado).
pc.run_setup("2026-03-02", rodar_outstanding=False)

## 4. Blocos individuais — rodar um fluxo isolado
Cada chamada é um CLI. Descomente o que precisar.

In [ ]:
X = "2026-07-02"
Xant = pc.dia_util_anterior(X).isoformat()
print("X =", X, "| X-1u =", Xant)

In [ ]:
# --- SCRAPING (rede) ---
pc.run_step("scrape_b3_boletim", "--start", Xant, "--end", X)
# pc.run_step("scrape_anbima_debentures", "--date", X)
# pc.run_step("scrape_anbima_cri_cra", "--date", X)
# pc.run_step("scrape_fianalytics_planilha")
# pc.run_step("scrape_anbima_data_ativos", "--start", Xant, "--end", X)
# pc.run_step("scrape_anbima_ntnb", "--start", Xant, "--end", X)
# pc.run_step("scrape_b3_curva_di", "--date", X)

In [ ]:
# --- CALCULO (local) ---
# pc.run_step("calc_taxa_negocios", "--date", X)
# pc.run_step("filtrar_trades", "--date", X)
# pc.run_step("calc_spread_anbima", "--date", X)
# pc.run_step("match_referencias")
# pc.run_step("calc_spread_over", "--date", X)
# pc.run_step("gerar_relatorio_credito")